# 时间与空间核 CDF

读取全 neural 模型的训练集核曲线，检查 CDF 随时间差、距离的变化。虚线为 CDF=1。输入由 `run_models.py` 默认生成；抽样仅用于绘图，不改变模型评估。

改编自 [zhancx-edu/NeuralKernelFunction 的 plot CDF.ipynb](https://github.com/zhancx-edu/NeuralKernelFunction/blob/febc15aefc316cf128ea217bbd60321c88fbaab1/plot%20CDF.ipynb)，适配本仓库 PyTorch 输出。

在仓库根目录启动 Jupyter，先安装 `requirements-plotting.txt`。修改下面的参数单元格后，从上到下执行。结果保存到 `fig/`；本 Notebook 不启动训练。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "run_models.py").is_file():
    raise FileNotFoundError("请在 ST-NKF-pytorch 仓库根目录启动 Jupyter / nbconvert。")

In [ ]:
DATASETS = ["SCEDC_20", "SCEDC_25", "SCEDC_30", "ComCat", "SaltonSea", "SanJac", "WHITE"]
RESULT_DIR = ROOT / "results"
FIG_DIR = ROOT / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_STEP = 1000
DPI = 600
SHOW_FIGURES = True

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.linewidth": 0.8})

In [ ]:
def load_curve(dataset, tag, x_name, y_name):
    if SAMPLE_STEP < 1:
        raise ValueError("SAMPLE_STEP 必须是正整数。")
    paths = [RESULT_DIR / f"{dataset}_{tag}_{name}.npy" for name in (x_name, y_name)]
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；请先运行对应模型，且不要使用 --no-patterns。")
    x, y = [np.load(path, mmap_mode="r") for path in paths]
    if x.ndim != 1 or y.ndim != 1 or x.shape != y.shape or x.size == 0:
        raise ValueError(f"{dataset}: 核曲线必须是长度相同的非空一维数组。")
    # 在不载入完整大数组的情况下抽样，并保留末点以显示尾部。
    indices = np.unique(np.r_[np.arange(0, x.size, SAMPLE_STEP), x.size - 1])
    xs, ys = np.asarray(x[indices]), np.asarray(y[indices])
    if not (np.isfinite(xs).all() and np.isfinite(ys).all()):
        raise ValueError(f"{dataset}: 曲线包含非有限值，请检查训练输出。")
    return xs, ys

def save_figure(fig, stem, pdf=False):
    fig.savefig(FIG_DIR / (stem + ".png"), dpi=DPI, bbox_inches="tight")
    if pdf:
        fig.savefig(FIG_DIR / (stem + ".pdf"), bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    print("Saved:", FIG_DIR / (stem + ".png"))

def plot_cdf(kind):
    if not DATASETS:
        raise ValueError("DATASETS 不能为空。")
    rows = (len(DATASETS) + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(6.8, 2.1 * rows), squeeze=False,
                             constrained_layout=True)
    axes = axes.ravel()
    x_name = "dis_t_train" if kind == "temporal" else "dis_xy_train"
    for i, dataset in enumerate(DATASETS):
        x, y = load_curve(dataset, "neural_neural_neural", x_name, f"cdf_{kind}_train")
        ax = axes[i]
        ax.plot(x, y, color="black", lw=1.5)
        ax.axhline(1, color="tab:red", ls="--", lw=1)
        ax.set_xlim(left=0)
        ax.set_ylim(0, 1.02)
        ax.set_xlabel("Time lag (days)" if kind == "temporal" else "Distance (km)")
        ax.set_ylabel("Temporal CDF" if kind == "temporal" else "Spatial CDF")
        ax.set_title(f"({chr(97 + i)}) {dataset}", loc="left", fontsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    for ax in axes[len(DATASETS):]:
        fig.delaxes(ax)
    save_figure(fig, "NKF_Temporal_CDF" if kind == "temporal" else "NKF_Spatial_CDF", pdf=True)

In [ ]:
plot_cdf("temporal")
plot_cdf("spatial")